In [ ]:
# Block 0 — setup. Code from GitHub onto the VM disk; data + embeddings on Drive.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# numpy/scipy/torch are filtered out: Colab's torch is built against its own numpy
!grep -viE '^(torch|torchvision|torchaudio|nvidia-|numpy|scipy|#)' requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt

In [ ]:
# Block 1 — ONE ARM of the frozen pan-cancer dataset, the shared embedding cache, evaluation sets, settings.
# Change ARM (and FOLD) here and nothing else: experiment ids carry the arm prefix, so arms never mix in the results store.
# Every experiment block below depends on THIS cell and on nothing else: they communicate only through the results store on Drive.
# The runner scores every non-train row, so tables are restricted to OWN (the arm's own test) and the other diseases are
# read through SETS (rows similar to the arm's train are already removed from them).
import pandas as pd
from src import config, data, embed, results, runner, pancancer_split as ps

NAME = "pancancer_cedar_v1"
ARM, FOLD = "colorectal", 0            # colorectal | lung | melanoma | pooled
PFX = ps.ARM_PREFIX[ARM]
IDS = {k: f"{PFX}_{k}" for k in ("G_baselines", "D_plm", "E_clf", "H_weight", "I_leak")}

df = ps.build_arm(config.DATA_DIR, NAME, ARM, FOLD)         # rebuilt from manifest.csv (deterministic), written and loaded
EID = df.attrs["emb_id"]                                    # the manifest-wide cache built by embeddings_pancancer.ipynb
MAXLEN = data.max_len(df)
SETS = ps.eval_uids(df)                                     # "<disease> (own test)", other core cancers, each other cancer, ...
OWN = ps.own_uids(df)
GROUPS = df.set_index("uid")["cluster_id"]                  # for the cluster bootstrap
SEEDS = (42, 43, 44)        # a few hundred test peptides at most: several seeds are needed to see the run-to-run spread
PLM = "prott5"              # feature set for the experiments, as in the paper
ESM = "esm2_650m"           # the ESM model used by the ESM baselines
FORCE = False               # True re-runs committed configurations
# Reduced mCNN for the proposal (paper default: windows 34/32/24/16 x 512 filters, ~10x more conv parameters).
# Restore the paper size by setting WINDOWS = [34, 32, 24, 16], FILTERS = 512, ONE_WINDOW = [34]. A new size is a new config hash,
# so nothing already committed is overwritten.
WINDOWS, FILTERS, ONE_WINDOW = [24, 16], 128, [24]

for plm in ("prott5", "esm2_650m", "esm1b"):
    if embed.is_cached(plm, EID):
        embed.stage(plm, EID)
    else:
        print(f"!! {plm} has no cache {EID} — run embeddings_pancancer.ipynb first")

got = ps.arm_counts(df)
print(f"{ARM} fold {FOLD}: train {got[0]} pos / {got[1]} neg | own test {got[2]} pos / {got[3]} neg "
      f"({got[2] / (got[2] + got[3]):.1%} positive)")
if FOLD == 0 and ARM in ps.DOCUMENTED:
    print("documented", ps.DOCUMENTED[ARM], "->", "OK" if got == ps.DOCUMENTED[ARM] else "!! DIFFERS: the manifest was rebuilt with other data or tools")
print("embedding cache:", EID, "| frame build id:", data.build_id(df), "| max_len:", MAXLEN, "| results:", config.RESULTS_DIR)

In [ ]:
# Block 2 — Experiment G: baselines (does the sequence carry a signal beyond trivial features?)
# Composition alone -> order (one-hot mCNN) -> pretraining (ESM + LogReg) -> local scanning on top of pretraining
# (ESM + CNN, ESM + mCNN). All three ESM rows share the same ESM-2 embeddings, so the gaps isolate what pooling,
# one window and several windows add. "length only" is a control: if peptide length alone separates the classes,
# a good AUC from a sequence model proves much less than it appears to.
cfgs = [
    dict(label="length only (LogReg)", features={"kind": "length"},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="Amino acid + LogReg", features={"kind": "aac"},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="One-hot + mCNN", features={"kind": "onehot"},
         model={"kind": "mcnn", "windows": WINDOWS, "filters": FILTERS}),
    dict(label="ESM + LogReg", features={"kind": "plm_pooled", "plm": ESM},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="ESM + CNN", features={"kind": "plm", "plm": ESM},
         model={"kind": "mcnn", "windows": ONE_WINDOW, "filters": FILTERS}),
    dict(label="ESM + mCNN", features={"kind": "plm", "plm": ESM},
         model={"kind": "mcnn", "windows": WINDOWS, "filters": FILTERS}),
]
runner.run_series(IDS["G_baselines"], cfgs, df=df, seeds=SEEDS, force=FORCE)
results.report(IDS["G_baselines"], OWN, SETS)

In [ ]:
# Block 3 — Experiment D: pre-trained language model features (paper Table 5, Fig 4a)
BEST_WINDOWS, BEST_FILTERS = WINDOWS, FILTERS
cfgs = [dict(label=name, features={"kind": "plm", "plm": plm},
             model={"kind": "mcnn", "windows": BEST_WINDOWS, "filters": BEST_FILTERS})
        for plm, name in [("prott5", "ProtTrans"), ("esm2_650m", "ESM-2"), ("esm1b", "ESM-1b")]]
runner.run_series(IDS["D_plm"], cfgs, df=df, seeds=SEEDS, force=FORCE)
results.report(IDS["D_plm"], OWN, SETS)

In [ ]:
# Block 4 — Experiment E: classifiers (paper Table 6, Fig 4b)
# KNN / RF / SVM take the mean-pooled embedding; the mCNN keeps per-residue input.
cfgs = [dict(label=n.upper(), features={"kind": "plm_pooled", "plm": PLM},
             model={"kind": "sklearn", "name": n}) for n in ("knn", "rf", "svm")]
cfgs.append(dict(label="mCNN", features={"kind": "plm", "plm": PLM},
                 model={"kind": "mcnn", "windows": WINDOWS, "filters": FILTERS}))
runner.run_series(IDS["E_clf"], cfgs, df=df, seeds=SEEDS, force=FORCE)
results.report(IDS["E_clf"], OWN, SETS)

In [ ]:
# Block 5 — Experiment H: class weighting. At this prevalence it is what moves sensitivity against precision.
tr = df[df["split"] == "train"]
print(f"{ARM} train prevalence: {tr['label'].mean():.1%} positive")
base = dict(features={"kind": "plm", "plm": PLM},
            model={"kind": "mcnn", "windows": WINDOWS, "filters": FILTERS})
cfgs = [dict(label="weighted BCE", train_cfg={"class_weight": True}, **base),
        dict(label="unweighted BCE", train_cfg={"class_weight": False}, **base)]
runner.run_series(IDS["H_weight"], cfgs, df=df, seeds=SEEDS, force=FORCE)
results.report(IDS["H_weight"], OWN, SETS)
print("Fixed 0.5 threshold on the own test — note how differently 0.5 behaves between the two")
display(results.fmt(results.table(IDS["H_weight"], subset=OWN, thr_mode="fixed")))

In [ ]:
# Block 6 — Experiment I: similarity leakage inside the training split.
# Random folds let near-duplicate peptides sit on both sides of a CV split; grouped folds keep each CD-HIT cluster together
# (the frame carries `cluster_id`). The leakage shows up in the CV table, not on the own test.
base = dict(features={"kind": "plm", "plm": PLM},
            model={"kind": "mcnn", "windows": WINDOWS, "filters": FILTERS})
cfgs = [dict(label="random folds", split_mode="stratified", **base),
        dict(label="similarity-controlled folds", split_mode="grouped", **base)]
runner.run_series(IDS["I_leak"], cfgs, df=df, seeds=SEEDS, force=FORCE)
results.report(IDS["I_leak"], OWN, SETS)

c = df.loc[df["split"] == "train", "cluster_id"].value_counts()
print(f"{int((df['split'] == 'train').sum())} training peptides fall into {len(c)} clusters; largest has {c.iloc[0]}")

In [ ]:
# Block 7 — uncertainty: AUC with a 95% CI that resamples whole clusters (the own-test peptides are not independent).
for k in ("G_baselines", "D_plm", "E_clf"):
    print(IDS[k])
    display(results.fmt(results.bootstrap_table(IDS[k], GROUPS, subset=OWN)))

In [ ]:
# Block 8 — report: every table rebuilt from the store. No training.
# This is what to run after a disconnect to see the whole state of the project.
TITLES = {IDS["G_baselines"]: "Baselines", IDS["D_plm"]: "PLM features", IDS["E_clf"]: "Classifiers",
          IDS["H_weight"]: "Class weighting", IDS["I_leak"]: "Similarity leakage"}
for exp_id, title in TITLES.items():
    t = results.table(exp_id, subset=OWN, thr_on=OWN)
    if t.empty:
        print(f"--- {title}: not run yet\n")
        continue
    print(f"--- {title} ({exp_id}) — {ARM} own test")
    display(results.fmt(t))
    results.roc(exp_id, subset=OWN, title=f"{title} — {ARM} own test")

runs = results.load_runs()
print(f"\n{len(runs)} committed runs in {config.RESULTS_DIR}")